In [1]:
%pip -q install duckdb huggingface_hub

import os
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':  f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':  f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':   f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:15} {n:>12,} rows')

dim_clients              104 rows
dim_content          519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily        78,835,655 rows
fact_query_90d     2,414,248 rows


# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shahzaib-Ali59/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**One row = one content item's (`content_hash_id`) daily performance within one client
(`client_hash_id`), for one calendar day**, in the raw `fact_content_daily_performance` table.

For this contract, I roll that up to: one row = one content item's **aggregated** performance
over one month. My working window is **March 2026** (`2026-03-01` to `2026-03-31`) — a
mid-panel month, not the final month (June 2026), which the warehouse README flags as a sealed
test month I should never use to develop label logic. I verify these claims with real queries
in Section 3.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

**Feature fields** (from `fact_content_daily_performance`, aggregated over March):
- `gsc_impressions` (summed) — how often the page appeared in search
- `gsc_clicks` (summed) — how often it was clicked
- `gsc_avg_position` (averaged) — average ranking position

**Feature fields** (from `fact_content_query_90d`):
- `content_visible_query_count` — how many distinct queries the page ranks for
- `rare_impressions_share` — share of impressions from rare/long-tail queries

**Label / proxy:** `is_declining` — whether a page's April impressions dropped more than 20%
versus its March impressions. This is a past→future label: March is the feature window, April
is the outcome window, so nothing about April leaks into the features.

**Context (not features):** `client_hash_id`, `access_profile` — used to understand which
clients are in my slice, never fed to a model directly.

**Excluded:** GA4-derived engagement columns, for any client where `has_ga4_access IS NOT TRUE`.
Including them would silently bias the feature set toward GA4-connected clients, since
non-GA4 clients would just show missing values rather than true zeros.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, availability) + features + the leakage trap

In [2]:
grain_check = con.sql(f"""
    SELECT content_hash_id, client_hash_id, COUNT(*) AS n_rows, COUNT(DISTINCT report_date) AS n_days
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
    GROUP BY 1, 2
    LIMIT 5
""").df()
print("If n_rows == n_days for each, the grain is confirmed: one row per content item per day.")
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

If n_rows == n_days for each, the grain is confirmed: one row per content item per day.


,content_hash_id,client_hash_id,n_rows,n_days
0,content_d0dff76c889de68f,client_62f4a7e64f5e0096,31,31
1,content_ac8663da7484669a,client_62f4a7e64f5e0096,31,31
2,content_39d7361b4945d504,client_62f4a7e64f5e0096,31,31
3,content_d49a012dcb924e31,client_62f4a7e64f5e0096,31,31
4,content_cec711b02f3bbde6,client_62f4a7e64f5e0096,31,31


In [3]:
span = con.sql(f"""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS first_day, MAX(report_date) AS last_day
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
""").df()
print(f"March 2026 slice: {span['n_rows'][0]:,} rows, spanning {span['first_day'][0]} to {span['last_day'][0]}")
span

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March 2026 slice: 9,841,378 rows, spanning 2026-03-01 00:00:00 to 2026-03-31 00:00:00


,n_rows,first_day,last_day
0,9841378,2026-03-01,2026-03-31


In [4]:
total = con.sql(f"""
    SELECT COUNT(*) FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
""").fetchone()[0]

available = con.sql(f"""
    SELECT COUNT(*)
    FROM {TABLES['fact_daily']} f
    JOIN {TABLES['dim_clients']} c ON f.client_hash_id = c.client_hash_id
    WHERE f.report_date BETWEEN '2026-03-01' AND '2026-03-31'
      AND c.has_gsc_access IS TRUE
""").fetchone()[0]

print(f"Total March rows: {total:,}")
print(f"Rows surviving has_gsc_access IS TRUE filter: {available:,} ({available/total*100:.1f}%)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total March rows: 9,841,378
Rows surviving has_gsc_access IS TRUE filter: 9,829,226 (99.9%)


### Five features (from March 2026)
1. `impressions_march` — knowable at the decision moment because it's already-observed historical traffic.
2. `clicks_march` — knowable because it's already-observed historical clicks.
3. `avg_position_march` — knowable because it's the page's already-measured average ranking.
4. `visible_queries` — knowable because the warehouse computes it from queries that already happened.
5. `rare_share` — knowable because it's derived from the same already-observed 90-day query window.

In [5]:
features = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS impressions_march,
           SUM(gsc_clicks) AS clicks_march,
           AVG(gsc_avg_position) AS avg_position_march
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 50
""").df()

qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count) AS visible_queries,
           ANY_VALUE(rare_impressions_share) AS rare_share
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

feature_frame = features.merge(qsignals, on='content_hash_id', how='left')
print(f"{len(feature_frame):,} content items in my March feature frame")
feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

116,114 content items in my March feature frame


,client_hash_id,content_hash_id,impressions_march,clicks_march,avg_position_march,visible_queries,rare_share
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1140.0,2.0,4.394234,8.0,0.028765
1,client_73cda7b4e4f265ea,content_05597932fe4da067,57.0,0.0,2.714744,NaN,NaN
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,149.0,0.0,6.481453,NaN,NaN
3,client_73cda7b4e4f265ea,content_05434271b257bb68,1421.0,6.0,6.320337,18.0,0.175435
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2770.0,16.0,4.459107,54.0,0.059806


In [6]:
april = con.sql(f"""
    SELECT content_hash_id, SUM(gsc_impressions) AS impressions_april
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-04-01' AND '2026-04-30'
    GROUP BY 1
""").df()

labeled = feature_frame.merge(april, on='content_hash_id', how='inner')
labeled['is_declining'] = (labeled['impressions_april'] < 0.8 * labeled['impressions_march']).astype(int)
print(f"{len(labeled):,} rows with a label, decline rate: {labeled['is_declining'].mean():.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

116,114 rows with a label, decline rate: 0.518


In [7]:
from sklearn.tree import DecisionTreeClassifier
import numpy as np

honest_features = ['impressions_march', 'clicks_march', 'avg_position_march', 'visible_queries', 'rare_share']
X_honest = labeled[honest_features].fillna(0)
y = labeled['is_declining']

tree_honest = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_honest, y)
print(f"HONEST score (March features only): {tree_honest.score(X_honest, y):.3f}")

# Now the trap: sneak in a column derived from the label period itself
labeled['LEAKY_pct_change'] = (labeled['impressions_april'] - labeled['impressions_march']) / labeled['impressions_march']
X_leaky = labeled[honest_features + ['LEAKY_pct_change']].fillna(0)

tree_leaky = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_leaky, y)
print(f"LEAKY score (with April-derived column): {tree_leaky.score(X_leaky, y):.3f}  <- jumps toward perfect")
print("\nThis is fake. LEAKY_pct_change is built FROM the label. Deleting it and keeping the honest score:")
print(f"Final, honest score: {tree_honest.score(X_honest, y):.3f}")

HONEST score (March features only): 0.672
LEAKY score (with April-derived column): 1.000  <- jumps toward perfect

This is fake. LEAKY_pct_change is built FROM the label. Deleting it and keeping the honest score:
Final, honest score: 0.672


## 4. Data limits

This slice can never tell me about clients whose `gsc_data_start` is after March 2026 —
newer clients would show partial or missing March data, making the panel unbalanced across
clients rather than a clean, comparable snapshot. Any conclusion I draw is about "clients
with a full March of history," not "all FlyRank clients." Additionally, ~116K of the 519K
total content items survived the `impressions_march >= 50` threshold — this feature frame
represents pages with meaningful traffic, not the full content catalog, so low-traffic or
brand-new pages aren't represented in this analysis.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.